Recompute criterion agreement from blinded expert ratings and saved model judgments, with case bootstrap confidence.


In [1]:
from pathlib import Path
import hashlib, itertools, json, re
import numpy as np
import pandas as pd
import openpyxl
from pathlib import Path
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'evaluations/human_and_model_validation/blind_protocols_60.csv').is_file() and (p / 'data').is_dir()))
BASE = ROOT / 'evaluations/human_and_model_validation'
OUT = BASE / 'expert_agreement'
OUT.mkdir(parents=True, exist_ok=True)
KEY = ['case_id', 'protocol', 'criterion']
H = ['expert_01', 'expert_02', 'expert_03', 'expert_04']
M = {'GPT-4.1': BASE / 'gpt41_judging/judge_gpt41/blind_protocols_60_gpt41_judge.csv', 'GPT-4.1 mini': BASE / 'gpt41_mini_judging/blind_protocols_60_gpt41_mini_judge.csv'}
B = 10000
SEED = 20261003

def read_csv(p):
    return pd.read_csv(p, sep=';', encoding='utf-8-sig')

def human(p, mapping):
    w = openpyxl.load_workbook(p, read_only=True, data_only=False)
    rows = []
    case = protocol = None
    for row in w['Review'].iter_rows(max_col=4):
        a, points, _, v = [c.value for c in row]
        label = '' if a is None else str(a)
        if (match := re.match('CASE\\s+(\\d+)\\s+\\u00b7', label)):
            case, protocol = (int(match[1]), None)
        elif (match := re.match('КРИТЕРИИ\\s*·\\s*ПРОТОКОЛ\\s+(\\d+)', label)):
            protocol = int(match[1])
        elif label.startswith('СВОДНАЯ'):
            protocol = None
        elif protocol is not None and isinstance(a, (int, float)):
            text = v.strip().lower() if isinstance(v, str) else v
            labels = {'да': 1, 'нет': 0}
            assert text in [None, '', *labels], (p, row[3].coordinate, v)
            rows.append([mapping[case], protocol, int(a), int(points), labels.get(text, np.nan), row[3].coordinate, case])
    w.close()
    d = pd.DataFrame(rows, columns=KEY + ['points', 'met', 'cell', 'case_number'])
    assert len(d) == 3408 and (not d.duplicated(KEY).any())
    return d

def model(p, blind):
    source = read_csv(p)
    rows = []
    assert len(source) == 240 and (not source.duplicated(['case_id', 'blind_slot']).any())
    assert set(source.case_id) == set(blind.case_id)
    for _, r in source.iterrows():
        ref = blind.loc[blind.case_id.eq(r.case_id)].iloc[0]
        text = r['winner_protocol'] if 'winner_protocol' in source else r['protocol']
        assert text == ref[f'protocol_{int(r.blind_slot)}']
        for v in json.loads(r.judge_details):
            assert type(v['met']) is bool
            axes = [t.removeprefix('axis:') for t in v['tags'] if t.startswith('axis:')]
            assert len(axes) == 1
            rows.append([r.case_id, int(r.blind_slot), int(v['id']), int(v['points']), int(v['met']), axes[0]])
    if 'protocol_text_matches_judged_source' in source:
        assert source.protocol_text_matches_judged_source.eq(True).all()
    d = pd.DataFrame(rows, columns=KEY + ['points', 'met', 'axis'])
    assert len(d) == 3408 and (not d.duplicated(KEY).any())
    return d

def kappa(ct):
    ct = np.asarray(ct, dtype=float)
    n = ct.sum(axis=-1)
    with np.errstate(divide='ignore', invalid='ignore'):
        po = (ct[..., 0] + ct[..., 3]) / n
        a = (ct[..., 2] + ct[..., 3]) / n
        b = (ct[..., 1] + ct[..., 3]) / n
        pe = a * b + (1 - a) * (1 - b)
        k = (po - pe) / (1 - pe)
    return (k, po, a, b, pe)

def counts(d, a, b, cases):
    v = d.dropna(subset=[a, b]).copy()
    v['code'] = v[a].astype(int) * 2 + v[b].astype(int)
    return pd.crosstab(v.case_id, v.code).reindex(index=cases, columns=range(4), fill_value=0).to_numpy()

def interval(x):
    return [float(v) for v in np.nanquantile(x, [0.025, 0.975])]

def pair(d, a, b, rng):
    v = d.dropna(subset=[a, b])
    cases = v.case_id.unique()
    ct = counts(v, a, b, cases)
    k, po, p1, p2, pe = kappa(ct.sum(axis=0))
    w = rng.multinomial(len(cases), np.full(len(cases), 1 / len(cases)), size=B)
    return dict(a=a, b=b, n=len(v), cases=len(cases), kappa=float(k), ci=interval(kappa(w @ ct)[0]), agreement=float(po), yes_a=float(p1), yes_b=float(p2), chance_agreement=float(pe), counts=ct.sum(axis=0).tolist())

def group_sums(d, complete):
    values = d[H]
    n = values.notna().sum(axis=1)
    yes = values.sum(axis=1)
    no = n - yes
    mask = n.eq(4) if complete else n.ge(2)
    out = d.loc[mask, ['case_id']].copy()
    n = n[mask]
    yes = yes[mask]
    no = no[mask]
    out['disagreement'] = 2 * no * yes / (n - 1)
    out['ratings'] = n
    out['yes'] = yes
    out['no'] = no
    out['items'] = 1
    out['pair_agreement'] = 1 - 2 * no * yes / (n * (n - 1))
    return out.groupby('case_id').sum()

def multi(s, kind):
    dis, total, yes, no, items, agree = np.moveaxis(np.asarray(s, dtype=float), -1, 0)
    if kind == 'alpha':
        return 1 - dis / total / (2 * yes * no / (total * (total - 1)))
    pe = (yes / total) ** 2 + (no / total) ** 2
    return (agree / items - pe) / (1 - pe)


In [2]:
def main():
    rng = np.random.default_rng(SEED)
    blind = read_csv(BASE / 'blind_protocols_60.csv')
    sel = read_csv(BASE / 'blind_protocols_60_selection.csv')
    assert len(blind) == 60 and blind.case_id.is_unique
    assert blind.case_id.tolist() == sel.case_id.tolist()
    mapping = dict(enumerate(blind.case_id, 1))
    humans = {n: human(BASE / f'expert_ratings/{n}_blind_protocols_60_ru_final.xlsx', mapping) for n in H}
    models = {n: model(p, blind) for n, p in M.items()}
    d = models['GPT-4.1'].drop(columns='met').copy()
    for name, ratings in {**humans, **models}.items():
        check = d.merge(ratings[KEY + ['points']], on=KEY, validate='one_to_one', suffixes=('', '_check'))
        assert len(check) == 3408 and check.points.eq(check.points_check).all()
        if 'axis' in ratings:
            check = d.merge(ratings[KEY + ['axis']], on=KEY, validate='one_to_one', suffixes=('', '_check'))
            assert check.axis.eq(check.axis_check).all()
        d = d.merge(ratings[KEY + ['met']].rename(columns={'met': name}), on=KEY, validate='one_to_one')
    coverage = {n: dict(n=int(r.met.notna().sum()), cases=int(r.loc[r.met.notna(), 'case_id'].nunique()), missing=int(r.met.isna().sum()), yes_rate=float(r.met.mean()), missing_cells=r.loc[r.met.isna(), ['case_number', 'protocol', 'criterion', 'cell']].to_dict('records')) for n, r in humans.items()}
    hl = [pair(d, n, m, rng) for n in H for m in M]
    hh = [pair(d, a, b, rng) for a, b in itertools.combinations(H, 2)]
    order = blind.case_id.tolist()
    weights = rng.multinomial(60, np.full(60, 1 / 60), size=B)
    macro = {}
    boot = {}
    contrast = {}
    for m in M:
        boot[m] = np.mean([kappa(weights @ counts(d, n, m, order))[0] for n in H], axis=0)
        rows = [r for r in hl if r['b'] == m]
        macro[m] = dict(kappa=float(np.mean([r['kappa'] for r in rows])), ci=interval(boot[m]), agreement=float(np.mean([r['agreement'] for r in rows])))
    for n in H:
        draws = [kappa(weights @ counts(d, n, m, order))[0] for m in M]
        points = [r['kappa'] for r in hl if r['a'] == n]
        contrast[n] = dict(mini_minus_full=points[1] - points[0], ci=interval(draws[1] - draws[0]))
    contrast['macro'] = dict(mini_minus_full=macro['GPT-4.1 mini']['kappa'] - macro['GPT-4.1']['kappa'], ci=interval(boot['GPT-4.1 mini'] - boot['GPT-4.1']))
    grouped = {}
    for label, kind, complete in [('krippendorff_alpha_available', 'alpha', False), ('fleiss_kappa_complete', 'fleiss', True), ('krippendorff_alpha_complete', 'alpha', True)]:
        sums = group_sums(d, complete)
        nc = len(sums)
        w = rng.multinomial(nc, np.full(nc, 1 / nc), size=B)
        grouped[label] = dict(value=float(multi(sums.sum().to_numpy(), kind)), ci=interval(multi(w @ sums.to_numpy(), kind)), items=int(sums['items'].sum()), ratings=int(sums.ratings.sum()), cases=nc, mean_pair_agreement=float(sums.pair_agreement.sum() / sums['items'].sum()))
    complete = d.dropna(subset=H)
    complete_hl = [pair(complete, n, m, rng) for n in H for m in M]
    strata = {}
    for field in ['axis', 'sign']:
        temp = d.assign(sign=np.where(d.points.gt(0), 'positive', 'penalty'))
        strata[field] = [dict(stratum=value, **pair(part, n, m, rng)) for value, part in temp.groupby(field) for n in H for m in M]
    pooled = {m: dict(zip(['kappa', 'agreement', 'yes_human', 'yes_llm', 'chance_agreement'], map(float, kappa(np.array([r['counts'] for r in hl if r['b'] == m]).sum(axis=0))))) for m in M}
    available = d[H].notna().sum(axis=1)
    votes = d[H].sum(axis=1)
    ties = votes.mul(2).eq(available)
    d['human_majority'] = np.where(available.ge(3) & ~ties, votes.mul(2).gt(available).astype(int), np.nan)
    consensus = [pair(d, 'human_majority', m, rng) for m in M]
    paths = [BASE / 'blind_protocols_60.csv', BASE / 'blind_protocols_60_selection.csv', *M.values(), *[BASE / f'expert_ratings/{n}_blind_protocols_60_ru_final.xlsx' for n in H]]
    result = dict(seed=SEED, bootstrap_replicates=B, coverage=coverage, human_llm=hl, human_pairs=hh, macro=macro, model_contrasts=contrast, multirater=grouped, complete_human_llm=complete_hl, strata=strata, pooled=pooled, consensus=consensus, consensus_ties=int(ties.sum()), llm_pair=pair(d, *M, rng), input_sha256={str(p.relative_to(BASE)): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths})
    (OUT / 'results.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    d.to_csv(OUT / 'aligned_ratings.csv', index=False, encoding='utf-8-sig')
    for label, records in [('human_llm', hl), ('human_pairs', hh), ('by_axis', strata['axis']), ('by_sign', strata['sign'])]:
        pd.DataFrame(records).to_csv(OUT / f'{label}.csv', index=False, encoding='utf-8-sig')
    compact = {k: v for k, v in result.items() if k not in ['coverage', 'strata', 'input_sha256', 'complete_human_llm']}
    compact['coverage'] = {n: {k: v for k, v in c.items() if k != 'missing_cells'} for n, c in coverage.items()}
    print(json.dumps(compact, ensure_ascii=False, indent=2))
    print('AXES')
    print(pd.DataFrame(strata['axis']).groupby(['stratum', 'b'])[['kappa', 'agreement']].mean().to_string())
    print('SIGNS')
    print(pd.DataFrame(strata['sign']).groupby(['stratum', 'b'])[['kappa', 'agreement']].mean().to_string())


In [3]:
main()


{
  "seed": 20261003,
  "bootstrap_replicates": 10000,
  "human_llm": [
    {
      "a": "expert_01",
      "b": "GPT-4.1",
      "n": 3406,
      "cases": 60,
      "kappa": 0.6531445266474764,
      "ci": [
        0.6152369754948563,
        0.6904429067230028
      ],
      "agreement": 0.8279506752789195,
      "yes_a": 0.4738696418085731,
      "yes_b": 0.42395772166764534,
      "chance_agreement": 0.5039740239410332,
      "counts": [
        1584,
        208,
        378,
        1236
      ]
    },
    {
      "a": "expert_01",
      "b": "GPT-4.1 mini",
      "n": 3406,
      "cases": 60,
      "kappa": 0.6150553171897923,
      "ci": [
        0.5754582486316271,
        0.6545233812937565
      ],
      "agreement": 0.8085731062830299,
      "yes_a": 0.4738696418085731,
      "yes_b": 0.448032883147387,
      "chance_agreement": 0.5027158387550691,
      "counts": [
        1510,
        282,
        370,
        1244
      ]
    },
    {
      "a": "expert_02",
      "b"